In [7]:
import pandas as pd
import numpy as np
import os
import time
from pathlib import Path

from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.endpoints import leaguedashplayerstats

# Makes nba_teams.csv

In [ ]:
from nba_api.stats.static import teams
import pandas as pd

#Get all NBA teams
nba_teams = pd.DataFrame(teams.get_teams())

#Select and rename columns
nba_teams = nba_teams[
    ["id", "abbreviation", "full_name"]
].copy()

nba_teams.columns = [
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "TEAM_NAME"
]

#Save dataset
nba_teams.to_csv("data/nba_teams.csv", index=False)

display(nba_teams)

,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME
0,1610612737,ATL,Atlanta Hawks
1,1610612738,BOS,Boston Celtics
2,1610612739,CLE,Cleveland Cavaliers
3,1610612740,NOP,New Orleans Pelicans
4,1610612741,CHI,Chicago Bulls
5,1610612742,DAL,Dallas Mavericks
6,1610612743,DEN,Denver Nuggets
7,1610612744,GSW,Golden State Warriors
8,1610612745,HOU,Houston Rockets
9,1610612746,LAC,Los Angeles Clippers


# Makes nba_champions csv

In [ ]:
#NBA Champions: 1996-97 through 2025-26

champions = [
    ("1996-97", "CHI"),
    ("1997-98", "CHI"),
    ("1998-99", "SAS"),
    ("1999-00", "LAL"),
    ("2000-01", "LAL"),
    ("2001-02", "LAL"),
    ("2002-03", "SAS"),
    ("2003-04", "DET"),
    ("2004-05", "SAS"),
    ("2005-06", "MIA"),
    ("2006-07", "SAS"),
    ("2007-08", "BOS"),
    ("2008-09", "LAL"),
    ("2009-10", "LAL"),
    ("2010-11", "DAL"),
    ("2011-12", "MIA"),
    ("2012-13", "MIA"),
    ("2013-14", "SAS"),
    ("2014-15", "GSW"),
    ("2015-16", "CLE"),
    ("2016-17", "GSW"),
    ("2017-18", "GSW"),
    ("2018-19", "TOR"),
    ("2019-20", "LAL"),
    ("2020-21", "MIL"),
    ("2021-22", "GSW"),
    ("2022-23", "DEN"),
    ("2023-24", "BOS"),
    ("2024-25", "OKC"),
    ("2025-26", "NYK")
]

#Create DataFrame
championship_history = pd.DataFrame(
    champions,
    columns=["SEASON", "TEAM_ABBREVIATION"]
)

#Add official NBA team IDs and names
championship_history = championship_history.merge(
    nba_teams,
    on="TEAM_ABBREVIATION",
    how="left",
    validate="many_to_one"
)

#Arrange columns
championship_history = championship_history[
    [
        "SEASON",
        "TEAM_ID",
        "TEAM_ABBREVIATION",
        "TEAM_NAME"
    ]
]

#Verify data
print("Total championships:", len(championship_history))
print("Missing values:")
print(championship_history.isnull().sum())

display(championship_history)

championship_history.to_csv("data/nba_champions.csv", index=False)


Total championships: 30
Missing values:
SEASON               0
TEAM_ID              0
TEAM_ABBREVIATION    0
TEAM_NAME            0
dtype: int64


,SEASON,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME
0,1996-97,1610612741,CHI,Chicago Bulls
1,1997-98,1610612741,CHI,Chicago Bulls
2,1998-99,1610612759,SAS,San Antonio Spurs
3,1999-00,1610612747,LAL,Los Angeles Lakers
4,2000-01,1610612747,LAL,Los Angeles Lakers
5,2001-02,1610612747,LAL,Los Angeles Lakers
6,2002-03,1610612759,SAS,San Antonio Spurs
7,2003-04,1610612765,DET,Detroit Pistons
8,2004-05,1610612759,SAS,San Antonio Spurs
9,2005-06,1610612748,MIA,Miami Heat


# Makes season wide stats

In [ ]:
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd
import time

seasons = [
    f"{year}-{str(year + 1)[-2:]}"
    for year in range(2010, 2026)
]

def download_team_category(seasons, measure_type):
    all_data = []

    for season in seasons:
        print(f"Downloading {measure_type}: {season}")

        for attempt in range(3):
            try:
                endpoint = leaguedashteamstats.LeagueDashTeamStats(
                    season=season,
                    season_type_all_star="Regular Season",
                    measure_type_detailed_defense=measure_type,
                    per_mode_detailed="PerGame",
                    timeout=60
                )

                df = endpoint.get_data_frames()[0]
                df["SEASON"] = season

                all_data.append(df)

                print(f"Success: {season}")
                time.sleep(2)
                break

            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(5)

        else:
            print(f"FAILED: {season}")

    if not all_data:
        raise RuntimeError(f"No {measure_type} data downloaded")

    return pd.concat(all_data, ignore_index=True)


team_base = download_team_category(
    seasons, "Base"
)

team_advanced = download_team_category(
    seasons, "Advanced"
)

team_opponent = download_team_category(
    seasons, "Opponent"
)


In [ ]:
#Merge base and advanced statistics
team_season_stats = team_base.merge(
    team_advanced,
    on=["TEAM_ID", "SEASON"],
    how="outer",
    suffixes=("", "_ADV")
)

#Merge opponent statistics
team_season_stats = team_season_stats.merge(
    team_opponent,
    on=["TEAM_ID", "SEASON"],
    how="left",
    suffixes=("", "_OPP")
)

#Add team abbreviations
team_season_stats = team_season_stats.merge(
    nba_teams[["TEAM_ID", "TEAM_ABBREVIATION"]],
    on="TEAM_ID",
    how="left"
)


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,...,OPP_AST_RANK,OPP_TOV_RANK,OPP_STL_RANK,OPP_BLK_RANK,OPP_BLKA_RANK,OPP_PF_RANK,OPP_PFD_RANK,OPP_PTS_RANK,PLUS_MINUS_RANK_OPP,TEAM_ABBREVIATION
0,1610612737,Atlanta Hawks,82,44,38,0.537,48.1,36.2,78.4,0.462,...,16,29,3,5,28,2,30,9,16,ATL
1,1610612737,Atlanta Hawks,66,40,26,0.606,49.0,36.8,81.0,0.454,...,14,9,7,10,23,11,25,6,6,ATL
2,1610612737,Atlanta Hawks,82,44,38,0.537,48.4,37.6,81.0,0.464,...,15,9,25,6,23,6,28,13,13,ATL
3,1610612737,Atlanta Hawks,82,38,44,0.463,48.4,37.3,81.6,0.458,...,24,10,26,12,26,11,24,15,17,ATL
4,1610612737,Atlanta Hawks,82,60,22,0.732,48.1,38.1,81.7,0.466,...,24,6,8,18,15,11,30,5,4,ATL


## Clean that jawn

In [ ]:
#Make a cleaned copy
team_season_stats_clean = team_season_stats.copy()

#Remove duplicate metadata columns
duplicate_columns = [
    "TEAM_NAME_ADV",
    "TEAM_NAME_OPP",
    "GP_ADV", "W_ADV", "L_ADV", "W_PCT_ADV", "MIN_ADV",
    "GP_OPP", "W_OPP", "L_OPP", "W_PCT_OPP", "MIN_OPP"
]

team_season_stats_clean = team_season_stats_clean.drop(
    columns=duplicate_columns,
    errors="ignore"
)

#Remove ranking columns
rank_columns = [
    col for col in team_season_stats_clean.columns
    if "RANK" in col
]

team_season_stats_clean = team_season_stats_clean.drop(
    columns=rank_columns
)

#Move identifying columns to the front
first_columns = [
    "SEASON",
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "TEAM_NAME"
]

remaining_columns = [
    col for col in team_season_stats_clean.columns
    if col not in first_columns
]

team_season_stats_clean = team_season_stats_clean[
    first_columns + remaining_columns
]

#Preview
display(team_season_stats_clean.head())

#Save cleaned dataset
team_season_stats_clean.to_csv(
    "data/team_season_stats_clean.csv",
    index=False
)

print("Rows:", len(team_season_stats_clean))
print("Columns:", len(team_season_stats_clean.columns))
print("Seasons:", team_season_stats_clean["SEASON"].nunique())
print("Teams per season:")
print(team_season_stats_clean.groupby("SEASON")["TEAM_ID"].nunique())

,SEASON,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,...,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,PLUS_MINUS_OPP
0,2010-11,1610612737,ATL,Atlanta Hawks,82,44,38,0.537,48.1,36.2,...,40.6,21.4,12.5,6.7,4.2,4.2,18.5,19.0,95.8,0.8
1,2011-12,1610612737,ATL,Atlanta Hawks,66,40,26,0.606,49.0,36.8,...,42.3,20.5,15.1,7.5,4.7,4.6,19.1,17.8,93.2,-3.4
2,2012-13,1610612737,ATL,Atlanta Hawks,82,44,38,0.537,48.4,37.6,...,43.7,22.3,15.2,8.4,4.3,4.5,18.8,18.0,97.5,-0.4
3,2013-14,1610612737,ATL,Atlanta Hawks,82,38,44,0.463,48.4,37.3,...,43.5,23.2,15.2,8.2,4.3,4.0,20.0,19.2,101.5,0.5
4,2014-15,1610612737,ATL,Atlanta Hawks,82,60,22,0.732,48.1,38.1,...,43.6,23.5,16.1,7.4,4.9,4.6,19.6,17.8,97.1,-5.4
